# Title: RAGCHECKER: A Fine-grained Framework for Diagnosing Retrieval-Augmented Generation

#### Members' Names: Amadike Lilian, Jeff Ogakwu, Philemon David

####  Emails: camadike@torontomu.ca, david.philemon@torontomu.ca, jeff.ogakwu@torontomu.ca

# Introduction:

#### **Problem Description:**

Retrieval-Augmented Generation (RAG) systems improve large language models by retrieving external documents and using them to generate answers. Although RAG can produce more factual and context-aware responses than standalone language models, evaluating its performance is difficult. Since a RAG system has both a retriever and a generator, errors in the final response may come from retrieving irrelevant information, missing useful information, or generating unsupported claims from the retrieved context. The RAGChecker paper identifies this as a major challenge in RAG evaluation, especially for long-form responses where coarse metrics do not capture fine-grained errors.


#### **Context of the Problem:**

This problem is important because RAG systems are now used in applications where factual accuracy and reliability matter, such as question answering and knowledge-based assistance. If evaluation is weak, it becomes hard to understand whether poor performance is caused by retrieval failure or generation failure. A more detailed evaluation is therefore necessary for diagnosing system behavior and improving the overall quality of responses.


#### **Limitation About other Approaches:**

Earlier approaches often rely on metrics such as BLEU, ROUGE, recall@k, or other coarse response-level measures. These approaches do not clearly separate retriever errors from generator errors, and they are not detailed enough to assess claim-level correctness in long answers.


#### **Solution:**

To overcome these challenges, we introduce RAGCHECKER, an innovative evaluation framework designed for detailed analysis of both retrieval and generation processes. RAGCHECKER is based onclaim-level entailment checking which involves operations of extracting claims from the response and ground truth answer and checking them against other texts. This approach enables fine-grained evaluation instead of response-level assessment. RAGCHECKER processes the user query, retrieved context, response, and ground truth answer, producing a suite of metrics:

- **Overall Metrics** to provide a holistic view of the system performance, assessing the overall quality of the generated responses.
-  **Diagnostic Retriever Metrics** to evaluate the effectiveness of the retriever, identifying its strengths and weaknesses in finding relevant information from the knowledge base.
- **Diagnostic Generator Metrics** to assess the performance of the generator, diagnosing how well the generator utilizes the retrieved context, handles noisy information, and generates accurate and faithful responses.

# Background


| Reference |Explanation |  Dataset/Input |Weakness
| --- | --- | --- | --- |
| Lewis et al. (RAG, 2020) [1]| They introduced the original RAG framework combining dense retrieval with sequence generation for knowledge-intensive tasks | Open-domain QA datasets (e.g., Natural Questions, Wikipedia corpus) | Focuses on model performance, not detailed evaluation of RAG components
| Es et al. (RAGAS) [2]| They evaluated RAG systems using metrics such as answer relevance, context relevance, and groundedness via LLM scoring | QA datasets with generated responses | Coarse-grained; cannot separate retriever vs generator errors
| Ferrara et al. (TruLens) [3] | They proposed the RAG Triad (context relevance, groundedness, answer relevance) for evaluating RAG pipelines| Query, retrieved context, and generated responses | Relies on LLM judgments; lacks fine-grained diagnostic capability
| Chen et al. (RGB Benchmark) [4] | They evaluated generator robustness in RAG systems across tasks like noise handling and information integration| Manually constructed QA datasets with perturbed contexts | Focuses only on generator; does not evaluate retrieval component
| Saad-Falcon et al. (ARES) [5] | They automated a framework for evaluating RAG systems using learned scoring models| QA datasets with generated outputs | Limited interpretability; focuses on overall scores rather than detailed error analysis
| Ru et al. (RAGChecker, 2024) [6] | They introduced claim-level evaluation to assess both retriever and generator using metrics like claim recall, context precision, and faithfulness| ClapNQ, RobustQA, LoTTE, KIWI, NovelQA | Computationally expensive; requires claim extraction and entailment models


# Methodology

## The Original Paper's Methodology

The original RAGChecker paper presents a complete framework for evaluating Retrieval-Augmented Generation (RAG) systems. Instead of only checking whether a final answer is correct, the paper evaluates both parts of the pipeline separately:

1. **Retriever** – finds relevant documents  
2. **Generator** – uses retrieved documents to produce the final answer  

The objective was to diagnose where system errors come from: poor retrieval, poor generation, or both.

---

### 1. RAG Systems Evaluated in the Paper

The authors built **8 RAG systems** by combining two retrievers with four generators.

### Retrievers

- **BM25**  
  A sparse keyword-based retriever that ranks documents using term matching.

- **E5-Mistral**  
  A dense semantic retriever that converts text into embeddings and retrieves documents based on meaning similarity.

### Generators

- **GPT-4**
- **Llama3-70B**
- **Mixtral-8x7B**
- **Command-R**

### Final 8 Systems

| Retriever | Generator |
|----------|-----------|
| BM25 | GPT-4 |
| BM25 | Llama3-70B |
| BM25 | Mixtral-8x7B |
| BM25 | Command-R |
| E5-Mistral | GPT-4 |
| E5-Mistral | Llama3-70B |
| E5-Mistral | Mixtral-8x7B |
| E5-Mistral | Command-R |

This setup allowed the authors to compare how retrieval type and generator choice affect final RAG quality.

---

### 2. Evaluation Dataset

The paper evaluated the systems on a benchmark containing:

- **4,162 queries**
- **10 knowledge-intensive domains**

Examples included factual question answering, open-domain retrieval, and information-seeking tasks.

Each sample included:

- User query  
- Retrieved context  
- Generated response  
- Ground truth answer

---

### 3. Claim-Level Evaluation Process

Traditional metrics score entire responses, but the paper argued that long answers often contain both correct and incorrect statements.

To solve this, RAGChecker evaluates at the **claim level**.

### Step A: Claim Extraction

The framework breaks both:

- Generated responses  
- Ground truth answers  

into smaller factual claims.

Example:

Response:  
*Toronto is in Ontario and is the capital of Canada.*

Claims:

- Toronto is in Ontario  
- Toronto is the capital of Canada

This creates fine-grained factual units for evaluation.

---

### Step B: Entailment / Verification Checking

Each claim is checked against different sources to determine whether it is:

- Supported  
- Contradicted  
- Not supported

The framework performs four core comparisons:

#### 1. Answer → Response

Checks whether claims from the ground truth answer appear correctly in the generated response.

Used for:

- Precision
- F1

#### 2. Response → Answer

Checks whether generated claims are correct compared with the reference answer.

Used for:

- Recall
- F1

#### 3. Retrieved Context → Answer

Checks whether retrieved documents contain the information needed to answer the question.

Used for:

- Claim Recall
- Context Precision

#### 4. Retrieved Context → Response

Checks whether generated claims are supported by the retrieved documents.

Used for:

- Faithfulness
- Hallucination
- Self-Knowledge

---

### 4. Metrics Computed

The framework computes three categories of metrics.

---

### A. Overall Metrics

Measure final answer quality.

- **Precision** – how much of the answer is correct  
- **Recall** – how much correct information was included  
- **F1-score** – balance between precision and recall  

---

### B. Retriever Metrics

Measure document retrieval quality.

- **Claim Recall** – whether needed information was retrieved  
- **Context Precision** – proportion of retrieved documents that were useful  

---

### C. Generator Metrics

Measure generation behavior.

- **Faithfulness** – answer supported by retrieved context  
- **Hallucination** – unsupported generated claims  
- **Self-Knowledge** – model answers from internal memory instead of context  
- **Context Utilization** – how well retrieved documents were used  
- **Noise Sensitivity (Relevant)** – confusion caused by relevant noise  
- **Noise Sensitivity (Irrelevant)** – confusion caused by irrelevant noise  

---

### 5. Aggregated System Comparison

Metric scores were averaged across all queries.

This enabled ranking and comparison of all 8 RAG systems to determine which retriever–generator combinations performed best.

---

### 6. Meta-Evaluation Against Human Judgment

The paper also tested whether RAGChecker metrics align with human preferences.

Human evaluators compared system outputs based on:

- Correctness  
- Completeness  
- Overall quality  

The authors then measured how well RAGChecker rankings matched human judgments.

This showed stronger diagnostic ability than coarse response-level metrics.

---

### Summary

The original paper built 8 RAG systems using sparse and dense retrievers with multiple LLM generators, then evaluated them using a claim-level framework that measures retrieval quality, generation quality, and alignment with human judgment.

The original methodology evaluates not only whether a final answer is correct, but also whether the right documents were retrieved and whether the generator used them faithfully.

---

## Our Project's Methodology 

This project implements a Retrieval-Augmented Generation (RAG) pipeline combined with a fine-grained evaluation framework inspired by RAGChecker. The methodology follows a structured sequence of steps, beginning with dataset preparation and ending with comparative performance analysis across multiple system configurations. The entire process can be understood as a sequence of structured steps:

![RAGChecker Methodology](https://drive.google.com/uc?export=view&id=1kViD1G5kwDRaGZFAK7cG0llxjyvhdbB4)

**Figure:** RAGChecker methodology pipeline showing dataset loading, parsing, corpus preparation, retrieval using FAISS and BM25, response generation, system construction, evaluation using claim-level metrics (precision, recall, claim recall, context precision, faithfulness, hallucination), and final analysis.
#

###**Step 1: Dataset Loading**

Two question-answering datasets are used for evaluation: NovelQA and ClapNQ. NovelQA is loaded from locally cached JSON files, while ClapNQ is accessed through the Hugging Face datasets library. In addition, the LoTTE science corpus is loaded using ir_datasets and serves as the external knowledge base for retrieval. This setup ensures a clear separation between evaluation queries and the document corpus used for retrieval.

#

###**Step 2: Dataset Parsing and Structuring**

The raw datasets are transformed into a consistent format. For NovelQA, each sample is parsed to extract the question, ground-truth answer, and supporting evidence. For ClapNQ, the query and reference answer are extracted and stored in the same structure. This standardization ensures compatibility across datasets and simplifies downstream processing.

#

###**Step 3: Retrieval Corpus Preparation**

The LoTTE corpus is processed into a structured document collection. Each document is assigned an identifier and stored as text, forming the retrieval corpus. This corpus is then prepared for efficient search during the retrieval stage.

#

###**Step 4: Retrieval Using FAISS and BM25**

Two retrieval strategies are implemented to obtain relevant context for each query:

Dense Retrieval (FAISS): Documents are embedded using a SentenceTransformer model (all-MiniLM-L6-v2) and indexed using FAISS. Queries are embedded into the same vector space, and the top-
𝑘
k most similar documents are retrieved based on semantic similarity.
Sparse Retrieval (BM25): Documents are tokenized and indexed using BM25, which retrieves documents based on lexical matching between query terms and document content.

These approaches enable comparison between semantic and keyword-based retrieval within the same pipeline.

#

###**Step 5: Response Generation**

Retrieved documents are passed to a generator to produce answers. Two generation strategies are used:

- Extractive baseline: returns answers directly from retrieved content
- GPT-based generator: uses a large language model to generate responses conditioned on the retrieved context

This enables comparison between non-generative and generative approaches.

#

###**Step 6: Construction and Formatting of RAG Systems**

Four RAG system configurations are constructed by combining retrievers and generators:

1.) FAISS + Extractive

2.) BM25 + Extractive

3.) FAISS + GPT

4.) BM25 + GPT

Each system follows the same pipeline but differs in how context is retrieved and how responses are generated. The outputs of each system, including generated responses and retrieved documents, are then converted into a standardized format compatible with the RAGChecker evaluation framework. This step ensures seamless integration between the implemented pipeline and the evaluation process.

#

###**Step 7: Evaluation Using RAGChecker**

Each system is evaluated on both NovelQA and ClapNQ datasets using a fine-grained, claim-level evaluation approach. The evaluation decomposes responses into individual claims and compares them against ground-truth answers and retrieved documents.

The following metrics are computed:

- Precision
- Recall
- f1 Score
- Claim Recall
- Context Precision
- Context Utilization
- Faithfulness
- Hallucination
- Self Knowledge
- Noise Sensitivity (Relevant)
- Noise Sensitivity (Irrelevant)

![RAGChecker Evaluation Metrics](https://drive.google.com/uc?export=view&id=1vjuKLckyxvGM486sqmwGZ1ek2ToE73e9)

**Figure:** RAGChecker evaluation metrics used in this project. The metrics include overall performance measures (precision, recall and f1), retriever metrics (claim recall and context precision), and generator metrics (context precision, faithfulness, hallucination, self knowledge, noise sensitivity for relevant and irrelevant), providing a fine-grained analysis of RAG system behavior.

These metrics provide detailed insights into both overall system performance and the behavior of individual components.

#

###**Step 8: Aggregation of Results**

Evaluation results from NovelQA and ClapNQ are combined by averaging the metric values across datasets. This provides a more robust and generalized comparison of system performance.

#

###**Step 9: Analysis and Comparison**

The final step involves analyzing the performance of all four RAG systems.

![RAG System Analysis and Comparison](https://drive.google.com/uc?export=view&id=1FrGbhr894uGLqORpxu8vDKiV0-mch-GU)

**Figure:** Comparison of the four RAG systems showing that FAISS improves recall, GPT improves precision, and BM25 reduces hallucination, highlighting trade-offs between coverage and reliability.

The comparison focuses on:

- Differences between FAISS and BM25 retrieval
- Improvements introduced by GPT-based generation
- Trade-offs between faithfulness and hallucination
- The relationship between retrieval quality and response accuracy

This analysis provides insights into how different design choices impact the effectiveness of RAG systems.


---

## Comparison of both Methodologies

To make the project practical within the available time and computational resources, we implemented a simplified version of the original RAGChecker framework while preserving its main evaluation objective.

### What We Kept from the Original Paper

The following core ideas from the paper were maintained:

- Evaluation of complete RAG pipelines (retriever + generator)
- Comparison of multiple RAG system configurations
- Use of benchmark question-answer datasets
- Claim-level evaluation using the RAGChecker framework
- Measurement of retrieval quality and generation quality separately

### What We Changed

#### 1. Reduced Number of Systems

The original paper evaluates 8 RAG systems using multiple retrievers and generators.

Our project evaluated **4 systems**:

- FAISS + Extractive
- BM25 + Extractive
- FAISS + GPT
- BM25 + GPT

This still allows comparison between sparse vs dense retrieval and simple vs LLM-based generation.

---

#### 2. Different Dense Retriever

The paper uses **E5-Mistral embeddings** as a strong dense retriever.

Our project used **FAISS with SentenceTransformer embeddings** because it is lighter, easier to run locally, and suitable for semantic retrieval experiments.

---

#### 3. Smaller Datasets

The original paper evaluates over **4,000+ queries across 10 domains**.

Our implementation used subsets of:

- NovelQA  
- CLAPNQ

This reduced runtime while still allowing meaningful evaluation across different question styles.

---

#### 4. Simpler Generators

The paper compares several advanced large language models.

Our project used:

- **Extractive baseline generator** (returns retrieved context summary)
- **GPT-based generator** for stronger natural language answers

This allowed comparison between baseline generation and LLM-based generation.

---

#### 5. Direct Use of RAGChecker Module

Instead of re-implementing claim extraction and entailment checking manually, we used the official **RAGChecker Python module**, created by the original authors. 

This ensured consistency with the original framework while allowing us to focus on experimentation and analysis.

---

### Why These Changes Were Necessary

These simplifications were made due to:

- Limited compute resources
- API cost considerations
- Time constraints
- Need for manageable experimentation within project scope

---

### Summary

Although simplified, our implementation preserves the paper’s central idea: evaluating RAG systems in a fine-grained way by separately analyzing retrieval effectiveness, answer quality, and hallucination behavior. Overall, our approach preserves the core concept of fine-grained, claim-level evaluation proposed in the paper, while adapting it to a smaller, more practical experimental setup.



# Implementation



###**1. Setup (clone + install)**

In [ ]:
# Codes below to navigate to content folder, clone repo and install necessary dependencies

%cd /content

!git clone https://github.com/amazon-science/RAGChecker.git || echo "Repo exists"

%cd /content/RAGChecker
!pip install -e .

%cd /content

# Let pip resolve compatible versions automatically
!pip install -q sentence-transformers==2.7.0 peft==0.11.1 faiss-cpu rank-bm25 datasets

In [ ]:
# Inspect RAGChecker directory /RAGChecker
!ls /content/RAGChecker

### **2. Data Loading**

### We will download LoTTE using ir_datasets

In [ ]:
!pip install ir_datasets

Load LoTTE documents and querys (Science domain) - LoTTE will be as retrieval corpus (documents) not for query → answer evaluation

In [ ]:
import ir_datasets

dataset = ir_datasets.load("lotte/science/dev/search")

Inspect Data (Queries and Documents) in the LoTTE

In [ ]:
queries = list(dataset.queries_iter())[:20]
docs = list(dataset.docs_iter())[:2000]  # Limit size

# Prints the first query and document
print("Sample query:", queries[0])
print("Sample doc:", docs[0])

Login to Hugging Face and enter access token need for NovelQA and ClapNQ datasets

In [ ]:
from huggingface_hub import login

login()

### Download NovelQA (Question + Answer) - dataset 1

**Note**: We would get a DatasetGenerationCastError here but it is expected and harmless because NovelQA has inconsistent JSON schema. But overall:
- Dataset will download successfully
- JSON files (B70.json, B66.json, etc.) will be in cache

In [ ]:
from datasets import load_dataset

_ = load_dataset("NovelQA/NovelQA", split="train[:1]") #train-test split

Find where NovelQA data was stored so that we can use it

In [ ]:
import json
from pathlib import Path

# Define directories where dataset files may be stored
search_paths = [
    Path("/content"),                     # Colab working directory
    Path("/root/.cache/huggingface")     # Hugging Face cached datasets
]

all_json = []

# Search for JSON files containing dataset samples
for base in search_paths:
    if base.exists():  # Ensure directory exists before searching
        # Recursively find all JSON files starting with 'B'
        all_json.extend(list(base.rglob("B*.json")))

# Print total number of dataset files found
print("Found:", len(all_json))

### Load and Parse NovelQA dataset
We also extract evidence from NovelQA data, will be used to compare retrieved docs vs evidence & analyze retrieval quality.
The evaluation was conducted on a subset of NovelQA queries due to data accessibility constraints and the computational cost of LLM-based evaluation. Despite the limited size, the dataset provides sufficient diversity to analyze comparative trends across different RAG configurations

In [ ]:
novelqa_samples = []

# Loop through all JSON files containing dataset samples
for file in all_json:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)  # Load JSON content

    # Extract question-answer pairs from each file
    for q_id, q_data in data.items():
        question = q_data.get("Question", "").strip()  # Extract question text
        answer = q_data.get("Answer", "").strip()      # Extract ground-truth answer

        # Collect all supporting evidence texts
        evidence_texts = []
        for ev in q_data.get("Evidences", {}).values():
            evidence_texts.append(ev.get("Evidence", "").strip())

        # Combine all evidence into a single string
        evidence = " ".join([e for e in evidence_texts if e])

        # Only keep valid samples with both question and answer
        if question and answer:
            novelqa_samples.append({
                "query_id": q_id,     # Unique identifier
                "query": question,    # Input question
                "gt_answer": answer,  # Ground-truth answer
                "evidence": evidence  # Supporting context
            })

# Display first two samples for verification
print(novelqa_samples[:2])

### Load ClapNQ data from from PrimeQA (Question + Answer) - Dataset 2

In [ ]:
from datasets import load_dataset

ds = load_dataset("PrimeQA/clapnq")
len(ds["train"]) # Size of ClapNQ Dataset

In [ ]:
# Inspect ClapNQ data to confirm structure
print(ds)
print(type(ds["train"]))
print(ds["train"][0])

Structure the data loaded from ClapNQ properly, in the way the RAG systems expects

In [ ]:
# Select a small subset (100-200 samples) from the CLAPNQ dataset for evaluation
subset = ds["train"].select(range(200))  # Size = 200

clapnq_samples = []

# Loop through each sample in the subset
for i, item in enumerate(subset):

    question = item["input"].strip()  # Extract and clean the question

    answers = item.get("output", [])  # Get list of possible answers

    # Extract the first answer if available
    if answers and isinstance(answers, list):
        answer = answers[0].get("answer", "").strip()
    else:
        answer = ""

    # Keep only valid samples with both question and answer
    if question and answer:
        clapnq_samples.append({
            "query_id": str(i),   # Assign unique ID
            "query": question,    # Input question
            "gt_answer": answer   # Ground-truth answer
        })

# Print total number of processed samples
print("Total CLAPNQ samples:", len(clapnq_samples))

Prepare LoTTE data for FAISS embedding

In [ ]:
# Loop through the queries and store the ids and queries in an array
lotte_queries = [
    {"query_id": q.query_id, "query": q.text}
    for q in queries
]

# Loop through the documents and store the id and queries in an array
lotte_corpus = [
    {"doc_id": d.doc_id, "text": d.text}
    for d in docs
]

# Print first two 
print(lotte_queries[:2])
print(lotte_corpus[:2])

### **3. Retrieval**

We employ a lightweight dense retriever (FAISS) instead of more advanced models such as E5-Mistral used in the paper due to computational constraints, and Sparse retriever (BM25) as used in the research paper.  While stronger embedding models may improve retrieval performance, our focus is on comparing retrieval strategies and generation methods rather than optimizing the retriever itself. Therefore, the chosen setup is sufficient to demonstrate the relative strengths of sparse and dense retrieval within the RAG framework.

### Build FAISS Retriever

FAISS retrieves documents using semantic similarity by comparing embeddings instead of exact keywords.

In [ ]:
# Install + Imports

!pip install -q sentence-transformers faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer # Transformer that was used
import numpy as np
import faiss

Create Embeddings

In [ ]:
# SentenceTransformer converts text (sentences) into numerical vectors (embeddings) that capture their meaning.
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [d.text for d in docs]   # from LoTTE
doc_ids = [d.doc_id for d in docs]

embeddings = model.encode(texts, normalize_embeddings=True)
embeddings = np.array(embeddings).astype("float32")

Build FAISS index

In [ ]:
# Facebook AI Similarity Search
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS index size:", index.ntotal)

Retrival function (FAISS)

In [ ]:
def retrieve_faiss(query, top_k=5):
    # Convert query into embedding (vector) using SentenceTransformer
    # normalize_embeddings=True ensures cosine similarity works correctly
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    
    # Search FAISS index to retrieve top_k most similar documents
    # scores = similarity scores, indices = positions of matching documents
    scores, indices = index.search(q_emb, top_k)

    results = []
    # Loop through retrieved document indices
    for i, idx in enumerate(indices[0]):
        results.append({
            "doc_id": doc_ids[idx],      # Retrieve document ID
            "text": texts[idx],          # Retrieve document text
            "score": float(scores[0][i]) # Similarity score (higher = more relevant)
        })
        
    return results  # Return list of top_k relevant documents

### Build BM25 Retriever
BM25 (Best Matching 25) is a ranking algorithm that retrieves documents based on keyword matching, using term frequency and inverse document frequency to score relevance. BM25 is a keyword-based method, so it works directly on text without converting it into embeddings, unlike FAISS which requires SentenceTransformer.

In [ ]:
# Best Matching 25 (BM25) setup for keyword-based retrieval

from rank_bm25 import BM25Okapi
import re

# Tokenization function: converts text to lowercase and splits into words
def tokenize(text):
    return re.findall(r"\w+", text.lower())

# Apply tokenization to all documents in the corpus
# BM25 works on tokenized text (not embeddings)
tokenized_corpus = [tokenize(t) for t in texts]

# Initialize BM25 model with the tokenized corpus
# This builds the index used for keyword-based retrieval
bm25 = BM25Okapi(tokenized_corpus)

Retrival function (BM25)

In [ ]:
def retrieve_bm25(query, top_k=5):
    # Tokenize the query (BM25 works on keywords, not embeddings)
    tokens = tokenize(query)
    
    # Compute relevance scores between query tokens and all documents
    scores = bm25.get_scores(tokens)

    # Get indices of top_k highest scoring documents (sorted descending)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    # Collect top retrieved documents
    for idx in top_indices:
        results.append({
            "doc_id": doc_ids[idx],      # Document identifier
            "text": texts[idx],          # Document text
            "score": float(scores[idx])  # BM25 relevance score
        })
        
    return results  # Return top_k most relevant documents

### Test both FAISS AND BM25 Retrievers

FAISS uses semantic similarity (embeddings), while BM25 uses keyword-based matching (term frequency scoring).
#
In simple terms
- Keyword matching → looks at words
- Semantic similarity → looks at meaning

In [ ]:
test_query = "is sudan iv hydrophobic or hydrophilic?"

print("FAISS RESULTS:")
for r in retrieve_faiss(test_query, top_k=3):
    print("-", r["text"][:120])

print("\nBM25 RESULTS:")
for r in retrieve_bm25(test_query, top_k=3):
    print("-", r["text"][:120])

Key Insights from testing the two Retrievers

- Neither system retrieves the exact answer
- But both retrieve partially useful context

Why Results Aren’t Perfect

Because:

LoTTE = general corpus (StackExchange)

The query = specific chemistry question

So no guarantee exact answer exists
retrieval is approximate

**This tells us that Retrieval quality varies across methods, and relevant but incomplete context can still be retrieved even when exact answers are absent.**

### **4. RAG Pipeline**

### Now to build our 4 RAG Systems, we define:
Generator A = extractive baseline (No LLM)

Generator B = OpenAI GPT (With LLM)

In [ ]:
from openai import OpenAI
import os
from getpass import getpass

# Prompt user to enter API key if not already set in environment
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

# Initialize OpenAI client
client = OpenAI()

def generate_extractive(query, docs, max_chars=400):
    # Combine retrieved document texts into a single context
    context = " ".join([d["text"] for d in docs])
    
    # Return truncated context as a simple extractive baseline answer
    return f"Answer: Based on the retrieved information, {context[:max_chars]}"

def generate_gpt(query, docs, model_name="gpt-4o-mini"):
    # Format retrieved documents with labels for clarity
    context = "\n\n".join([f"[Doc {i+1}] {d['text']}" for i, d in enumerate(docs)])

    # Construct prompt to guide the model to use only retrieved context
    prompt = f"""
You are answering a question using only the retrieved context below.

Question:
{query}

Retrieved context:
{context}

Write a concise answer grounded only in the retrieved context.
If the context is insufficient, say so clearly.
""".strip()

    # Call GPT model to generate response
    response = client.chat.completions.create(
        model=model_name,
        temperature=0,  # Set to 0 for deterministic, factual responses
        messages=[{"role": "user", "content": prompt}]
    )

    # Extract and return generated answer
    return response.choices[0].message.content.strip()

### Build full RAG Pipeline

run_rag() function that supports both retrievers and both generators (baseline/ No LLM and GPT/ With specified LLM)

In [ ]:
def run_rag(query, retriever="faiss", generator="extractive", top_k=5):
    # Select retrieval method (FAISS = semantic, BM25 = keyword-based)
    if retriever == "faiss":
        docs = retrieve_faiss(query, top_k=top_k)
    elif retriever == "bm25":
        docs = retrieve_bm25(query, top_k=top_k)
    else:
        raise ValueError(f"Unknown retriever: {retriever}")

    # Select generation method (extractive baseline or GPT-based)
    if generator == "extractive":
        response = generate_extractive(query, docs)
    elif generator == "gpt":
        response = generate_gpt(query, docs)
    else:
        raise ValueError(f"Unknown generator: {generator}")

    # Return generated answer and retrieved documents for evaluation
    return response, docs

We defined the four RAG systems explicitly

| Retriever | Generator                |
| --------- | ------------------------ |
| FAISS     | Extractive (No LLM) |
| BM25      | Extractive (No LLM) |
| FAISS     | GPT                      |
| BM25      | GPT                      |


In [ ]:
systems = [
    {"name": "FAISS_Extractive", "retriever": "faiss", "generator": "extractive", "top_k": 5},
    {"name": "BM25_Extractive",  "retriever": "bm25",  "generator": "extractive", "top_k": 5},
    {"name": "FAISS_GPT",        "retriever": "faiss", "generator": "gpt",        "top_k": 5},
    {"name": "BM25_GPT",         "retriever": "bm25",  "generator": "gpt",        "top_k": 5},
]

### Build function to run the 4 RAG Systems ( code extracted from the github repo)
Used a smaller limit because GPT evaluation plus RAGChecker claim extraction has high computational cost

In [ ]:
def build_results_for_system(samples, retriever, generator, top_k=5, limit=5):
    results = []

    # Loop through a subset of samples (limit controls evaluation size)
    for sample in samples[:limit]:
        # Run full RAG pipeline (retrieval + generation)
        response, docs = run_rag(
            sample["query"],
            retriever=retriever,
            generator=generator,
            top_k=top_k
        )

        # Store all relevant information for evaluation
        results.append({
            "query_id": sample["query_id"],     # Unique query identifier
            "query": sample["query"],           # Input question
            "ground_truth": sample["gt_answer"],# Ground-truth answer
            "response": response,               # Generated answer
            "documents": docs,                  # Retrieved documents
            "evidence": sample.get("evidence", "")  # Optional supporting evidence
        })

    return results  # Return structured results for evaluation

### **5. Monkey Patch litellm**

### Set RAGChecker to use OpenAI instead of falling back to Bedrock

We do this because Bedrock will fail since we have no AWS credentials, so we used OpenAI which we have instead.

In [ ]:
import litellm

# Save original completion function
original_completion = litellm.completion

def force_openai_model(*args, **kwargs):
    # If no model is specified OR a non-OpenAI (e.g., bedrock) model is used,
    # override it with a default OpenAI model
    if "model" not in kwargs or "bedrock" in str(kwargs.get("model", "")):
        kwargs["model"] = "gpt-4o-mini"

    # Call the original completion function with updated parameters
    return original_completion(*args, **kwargs)

# Override litellm's completion function globally
# Ensures all calls use the specified OpenAI model
litellm.completion = force_openai_model

Load and view their container.py to identify the modules needed to wrap out results into RAGResult / RAGResults format, by uncommenting the code below

In [ ]:
!sed -n '1,200p' /content/RAGChecker/ragchecker/container.py

Add helper function extracted from code repo, to wrap our results into RAGResult / RAGResults

In [ ]:
# Add RAGChecker to python path to make sure import doesn't fail
import sys
sys.path.append("/content/RAGChecker")

In [ ]:
from ragchecker.container import RAGResults, RAGResult
from types import SimpleNamespace

def to_ragchecker_input(results):
    ragchecker_results = []

    for item in results:

        # Convert retrieved documents (dict format) into objects with attributes
        # RAGChecker expects objects with .doc_id and .text fields
        retrieved_context = [
            SimpleNamespace(doc_id=d["doc_id"], text=d["text"])
            for d in item["documents"]
        ]

        # Create RAGResult object for each query-response pair
        ragchecker_results.append(
            RAGResult(
                query_id=str(item["query_id"]),   # Unique query identifier
                query=item["query"],              # Input question
                gt_answer=item["ground_truth"],   # Ground-truth answer
                response=item["response"],        # Generated response
                retrieved_context=retrieved_context  # Retrieved documents
            )
        )

    # Wrap all results into RAGResults object required by RAGChecker
    return RAGResults(results=ragchecker_results)

### **6. Evaluation**
We will use the RAGChecker to evaluate the 4 RAG system on both datasets (NovelQA & ClapNQ)

Load and view the evaluator.py from the code repo by uncommenting the code below

This shows:

- How evaluation is actually done

- What functions/classes are used

In [ ]:
!sed -n '1,200p' /content/RAGChecker/ragchecker/evaluator.py

### Implement RAGChecker to evaluate each of the 4 RAG systems and get the fine-grained metrics

**NovelQA Dataset Evaluation Using RAGChecker (code adapted from their github repo)**

In [ ]:
from ragchecker import RAGChecker
import pandas as pd

# Initialize RAGChecker for claim-level evaluation
checker = RAGChecker()

all_eval_rows = []     # Stores summary results for each system
all_raw_evals = {}     # Stores full evaluation outputs

# Loop through each RAG system configuration
for sys in systems:
    print(f"Running {sys['name']} ...")

    # Run RAG pipeline (retrieval + generation) for current system
    sys_results = build_results_for_system(
        samples=novelqa_samples,
        retriever=sys["retriever"],
        generator=sys["generator"],
        top_k=sys["top_k"],
        limit=20   # Limit number of queries for evaluation efficiency
    )

    # Convert results into RAGChecker input format
    rag_input = to_ragchecker_input(sys_results)

    # Perform claim-level evaluation
    evaluation = checker.evaluate(rag_input)

    # Store raw evaluation output for detailed analysis
    all_raw_evals[sys["name"]] = evaluation

    # Flatten evaluation results into a dictionary for DataFrame
    if isinstance(evaluation, dict):
        row = {"System": sys["name"], **evaluation}
    else:
        # Handle case where evaluation is returned as an object
        row = {"System": sys["name"]}
        for attr in dir(evaluation):
            if not attr.startswith("_"):  # Ignore internal attributes
                val = getattr(evaluation, attr)
                if isinstance(val, (int, float, str)):
                    row[attr] = val

    # Add system results to list
    all_eval_rows.append(row)

# Convert all results into a DataFrame for easy comparison
results_df = pd.DataFrame(all_eval_rows)

# Display evaluation results table
results_df

Clean version of the Metrics table for NovelQA

In [ ]:
import pandas as pd

clean_rows = []

for system, eval_obj in all_raw_evals.items():

    overall = eval_obj["overall_metrics"]
    retriever = eval_obj["retriever_metrics"]
    generator = eval_obj["generator_metrics"]

    # Append all metric columns 
    clean_rows.append({
        "System": system,
        "Precision": round(float(overall["precision"]), 2),
        "Recall": round(float(overall["recall"]), 2),
        "F1 Score": round(float(overall["f1"]), 2),
        "Claim Recall": round(float(retriever["claim_recall"]), 2),
        "Context Precision": round(float(retriever["context_precision"]), 2),
        "Context Utilization": round(float(generator["context_utilization"]), 2),
        "Faithfulness": round(float(generator["faithfulness"]), 2),
        "Hallucination": round(float(generator["hallucination"]), 2),
        "Self Knowledge": round(float(generator["self_knowledge"]), 2),
        "Noise Sentivity (Relevant)": round(float(generator["noise_sensitivity_in_relevant"]), 2),
        "Noise Sentivity (Irrelevant)": round(float(generator["noise_sensitivity_in_irrelevant"]), 2)
    })

# Create and display final table
df_novelqa = pd.DataFrame(clean_rows)
df_novelqa

![NovelQA Metrics Table](https://drive.google.com/uc?export=view&id=19WDP4-AtwIHbl7f9TVoCp-wsRxhmcTDb)


**ClapNQ Dataset Evaluation Using RAGChecker**

In [ ]:
all_raw_evals_clapnq = {}

# Loop through each RAG system configuration
for sys in systems:

    print(f"Running {sys['name']} on CLAPNQ...")

    sys_results = []

    # Run RAG pipeline on each CLAPNQ sample
    for sample in clapnq_samples:
        response, docs = run_rag(
            sample["query"],
            retriever=sys["retriever"],   # Select retrieval method
            generator=sys["generator"]    # Select generation method
        )

        # Store results for evaluation
        sys_results.append({
            "query_id": sample["query_id"],
            "query": sample["query"],
            "ground_truth": sample["gt_answer"],
            "response": response,
            "documents": docs
        })

    # Convert results into RAGChecker-compatible format
    rag_input = to_ragchecker_input(sys_results)

    # Perform claim-level evaluation using RAGChecker
    evaluation = checker.evaluate(rag_input)

    # Store evaluation results for each system
    all_raw_evals_clapnq[sys["name"]] = evaluation

Clean version of the Metrics table for ClapNQ

In [ ]:
clean_rows_clapnq = []

for system, eval_obj in all_raw_evals_clapnq.items():

    overall = eval_obj["overall_metrics"]
    retriever = eval_obj["retriever_metrics"]
    generator = eval_obj["generator_metrics"]

    # Append all metric columns 
    clean_rows_clapnq.append({
        "System": system,
        "Precision": round(float(overall["precision"]), 2),
        "Recall": round(float(overall["recall"]), 2),
        "F1 Score": round(float(overall["f1"]), 2),
        "Claim Recall": round(float(retriever["claim_recall"]), 2),
        "Context Precision": round(float(retriever["context_precision"]), 2),
        "Context Utilization": round(float(generator["context_utilization"]), 2),
        "Faithfulness": round(float(generator["faithfulness"]), 2),
        "Hallucination": round(float(generator["hallucination"]), 2),
        "Self Knowledge": round(float(generator["self_knowledge"]), 2),
        "Noise Sentivity (Relevant)": round(float(generator["noise_sensitivity_in_relevant"]), 2),
        "Noise Sentivity (Irrelevant)": round(float(generator["noise_sensitivity_in_irrelevant"]), 2)
    })

# Create and display final table
df_clapnq = pd.DataFrame(clean_rows_clapnq)
df_clapnq

![ClapNQ Metrics Table](https://drive.google.com/uc?export=view&id=1ojtNuW8ZhGxCY5tFr2C8eBOiTZsbyxAe)

Average Performace across both the NovelQA and ClapNQ dataset

In [ ]:
# Use full metrics tables created earlier
# novelqa_full_df
# clapnq_full_df

# Set index to System for both
df_nq = df_novelqa.set_index("System")
df_cq = df_clapnq.set_index("System")

# Take average across datasets
df_avg = (df_nq + df_cq) / 2

df_avg

df_paper = df_avg.reset_index()

# Arrange all metrics
df_paper = df_paper[[
    "System",
    "Precision",
    "Recall",
    "F1 Score",
    "Claim Recall",
    "Context Precision",
    "Context Utilization",
    "Faithfulness",
    "Hallucination",
    "Self Knowledge",
    "Noise Sentivity (Relevant)",
    "Noise Sentivity (Irrelevant)"
]]

# Create grouped header 
df_paper.columns = pd.MultiIndex.from_tuples([
    ("RAG systems", ""),
    
    ("Overall", "Precision"),
    ("Overall", "Recall"),
    ("Overall", "F1"),
    
    ("Retriever", "Claim Recall"),
    ("Retriever", "Context Precision"),
    
    ("Generator", "Context Utilization"),
    ("Generator", "Noise Sensitivity (Relevant)"),
    ("Generator", "Noise Sensitivity (Irrelevant)"),
    ("Generator", "Hallucination"),
    ("Generator", "Self Knowledge"),
    ("Generator", "Faithfulness"),
])

# Round values
df_paper = df_paper.round(2)

# Display final table
df_paper

![Averaged Metrics Table](https://drive.google.com/uc?export=view&id=1Aniw1DGM0qmEHIb6_FmnhuI5oYg6PC0f)

![RAG Results Bar Chart](https://drive.google.com/uc?export=view&id=1RyR8ueVd5VUuWLR1Fsha9bLUC-go_d37)

### **7. Analysis**

### Analysis of Averaged RAG System Performance

### 1. Overall Performance (Precision, Recall & F1)

Across both datasets, GPT-based systems clearly outperform extractive baselines in overall answer quality. BM25_GPT achieves the highest average precision (30.65), closely followed by FAISS_GPT (29.65), while extractive systems remain significantly lower. This confirms that LLM-based generation substantially improves answer correctness by synthesizing information from retrieved context rather than directly copying text.

In terms of recall, FAISS_GPT performs best (9.30), followed by BM25_GPT (8.20). This suggests that FAISS retrieval provides broader semantic coverage, helping systems include more relevant claims in final responses.

F1-score values remain relatively low across all systems, with BM25_GPT (2.8) and FAISS_GPT (2.7) performing best. This is expected because claim-level precision and recall are strict metrics, where partially correct answers may still receive low overlap scores.

Overall, GPT-based systems dominate final answer quality, while extractive methods struggle despite stronger grounding.

---

### 2. Retriever Performance (Claim Recall & Context Precision)

Retriever metrics show that FAISS-based systems consistently achieve stronger retrieval coverage.

FAISS_GPT obtains the highest claim recall (10.00), indicating better retrieval of answer-supporting information. FAISS_Extractive follows closely (7.50), while BM25 systems remain lower.

This suggests that dense retrieval methods are more effective at capturing semantically relevant content, especially when queries require conceptual similarity rather than exact keyword matches.

Context precision is highest for FAISS_GPT (11.0), showing that FAISS retrieval not only captures more relevant claims, but also provides slightly more useful retrieved context overall.

BM25 systems remain competitive but appear more limited in coverage due to their lexical matching behavior.

These findings support the RAGChecker paper’s conclusion that retriever choice significantly influences downstream generation quality.

---

### 3. Generator Performance (Faithfulness, Hallucination & Self-Knowledge)

Generator metrics reveal an important trade-off between answer quality and factual reliability.

FAISS_Extractive achieves the highest faithfulness (38.05), followed by BM25_Extractive (32.60). This is expected because extractive systems rely directly on retrieved documents, making their outputs strongly grounded in context.

However, these same systems perform poorly in precision, meaning that highly faithful answers are not always the most useful or complete.

GPT-based systems achieve much stronger precision but lower faithfulness (≈14). This indicates that generative models improve answer quality through reasoning and summarization, but rely less directly on retrieved text.

Hallucination scores are lowest for BM25_Extractive (0.00) and FAISS_Extractive (0.40), confirming that extractive methods are safest and least likely to invent unsupported claims.

GPT systems show higher hallucination, especially BM25_GPT (8.65) and FAISS_GPT (8.60), reflecting the common trade-off between generation flexibility and factual grounding.

Self-knowledge is higher in extractive systems, while GPT systems show near-zero values in the averaged results, suggesting stronger dependence on retrieved context during evaluation.

---

### 4. Trade-offs Between FAISS and BM25

A clear pattern emerges between dense retrieval (FAISS) and sparse retrieval (BM25).

**FAISS** provides stronger semantic retrieval, resulting in:

- Higher recall  
- Higher claim recall  
- Better context precision  

This makes FAISS more suitable when queries require conceptual understanding or paraphrase matching.

**BM25** provides stronger lexical precision, resulting in:

- Highest overall precision (BM25_GPT)  
- Lower hallucination in extractive settings  
- Stable performance across datasets  

This makes BM25 effective when relevant documents contain strong keyword overlap with the query.

The trade-off can be summarized as:

**FAISS = Better semantic coverage**  
**BM25 = Better lexical precision and reliability**

---

### 5. Best Performing System

When averaging results across both datasets, **BM25_GPT** emerges as the strongest overall system, achieving:

- Highest precision (30.65)  
- Highest F1-score (2.8)  
- Strong competitive recall  
- Balanced performance across metrics  

FAISS_GPT remains a close second and performs best in retrieval-oriented metrics such as recall and claim recall.

This suggests:

- **BM25_GPT** is the best balanced practical system  
- **FAISS_GPT** is the best retrieval-focused system  

---

### 6. Key Insight: Generalization Across Datasets

Averaging across NovelQA and CLAPNQ reveals an important conclusion:

RAG system performance depends strongly on both retriever choice and generator type.

- GPT-based generation consistently improves final answer quality  
- FAISS improves semantic retrieval coverage  
- BM25 improves precision and output stability  
- Extractive systems maximize grounding but sacrifice answer usefulness  

These results demonstrate that no single component dominates all metrics. Instead, the best RAG system depends on the desired balance between:

- Accuracy  
- Completeness  
- Reliability  
- Hallucination control  

This supports the central argument of the RAGChecker paper: evaluating RAG systems requires fine-grained metrics that separately analyze retrieval and generation behavior.

# Conclusion and Future Directions

In this project, we implemented and evaluated multiple Retrieval-Augmented Generation (RAG) systems to study how different retrievers and generators influence final response quality. Four system configurations were constructed by combining two retrieval methods (**FAISS** and **BM25**) with two generation approaches (**extractive** and **GPT-based**). These systems were evaluated using the RAGChecker framework on two benchmark datasets: **NovelQA** and **CLAPNQ**.

The experimental results showed that RAG performance depends strongly on the interaction between retrieval and generation components rather than any single module alone. GPT-based systems consistently achieved the highest precision and overall answer quality, demonstrating the advantage of generative models in synthesizing information into coherent responses. In particular, **BM25_GPT** achieved the highest average precision across both datasets, while **FAISS_GPT** performed best in recall and claim recall, indicating stronger semantic retrieval coverage.

Retriever comparisons also revealed meaningful trade-offs. **FAISS**, as a dense semantic retriever, was more effective at capturing relevant information even when wording differed from the query. This improved recall and claim recall, but sometimes introduced noisier context. In contrast, **BM25**, as a sparse keyword-based retriever, produced more focused retrieval results and more stable outputs, especially on datasets with strong lexical overlap between query and source documents.

Generator metrics further highlighted the balance between answer quality and factual reliability. GPT-based generators produced significantly better answers than extractive baselines, but also introduced higher hallucination scores. Extractive systems, while limited in flexibility and completeness, achieved the strongest faithfulness and lowest hallucination because they remained closely tied to retrieved documents.

A key contribution of this project was the use of **fine-grained claim-level evaluation** rather than relying only on coarse metrics such as accuracy or exact match. By using metrics such as **precision, recall, claim recall, context precision, context utilization, faithfulness, hallucination, self-knowledge, and noise sensitivity**, it became possible to diagnose whether errors originated from the retriever or the generator. This directly supports the main objective of the original RAGChecker paper.

---

Although the implementation successfully reproduced the core ideas of the paper, some limitations remain. The original study evaluated eight RAG systems on a much larger benchmark across multiple domains, whereas this project used a smaller subset of datasets and four simplified system configurations. In addition, official dense retrievers such as E5-Mistral were replaced with FAISS-based embeddings for practical resource reasons.

---

For future work, several improvements are possible. Stronger embedding retrievers and reranking methods could be introduced to reduce noisy retrieval while preserving recall. More controlled prompting strategies or constrained generation methods could improve faithfulness and reduce hallucination. Evaluating additional datasets and larger-scale benchmarks would improve generalization analysis. Finally, implementing a manual version of the RAGChecker pipeline, including claim extraction and entailment verification, would provide deeper insight into the internal mechanics of claim-level evaluation.

Overall, this project demonstrated that effective RAG systems require a careful balance between retrieval quality, generation quality, and factual grounding. It also showed that detailed evaluation frameworks such as RAGChecker are essential for understanding system behavior and guiding future improvements.

# References:

[1]: D. Ru, L. Qiu, X. Hu, T. Zhang, P. Shi, S. Chang, C. Jiayang, C. Wang, S. Sun, H. Li, Z. Zhang, B. Wang, J. Jiang, T. He, Z. Wang, P. Liu, Y. Zhang, and Z. Zhang, “RAGChecker: A Fine-grained Framework for Diagnosing Retrieval-Augmented Generation,” arXiv preprint arXiv:2408.08067, 2024.

[2]: P. Lewis, E. Perez, A. Piktus, F. Petroni, V. Karpukhin, N. Goyal, H. Küttler, M. Lewis, W. Yih, T. Rocktäschel, S. Riedel, and D. Kiela, “Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks,” in Proc. Adv. Neural Inf. Process. Syst. (NeurIPS), 2020.

[3]: E. Es, J. James, L. Espinosa Anke, and S. Schockaert, “RAGAS: Automated Evaluation of Retrieval Augmented Generation,” in Proc. Conf. Empirical Methods in Natural Language Processing (EMNLP), 2023.

[4]: F. Ferrara, A. Pandey, and H. Chase, “TruLens: A Framework for Evaluating Large Language Model Applications,” arXiv preprint arXiv:2307.13970, 2023.

[5]: M. Saad-Falcon, D. Xu, and C. Callison-Burch, “ARES: An Automated Evaluation Framework for Retrieval-Augmented Generation Systems,” in Proc. Annual Meeting of the Association for Computational Linguistics (ACL), 2024.

[6]: J. Chen, H. Lin, X. Han, and L. Sun, “Benchmarking Large Language Models in Retrieval-Augmented Generation,” arXiv preprint arXiv:2309.01431, 2023.

# Appendix

### Final Comparison of Our Project with the Original RAGChecker Paper

**What was done in the original paper**

According to the original authors:

- Proposed RAGChecker, a fine-grained evaluation framework based on claim-level entailment
- Evaluated 8 RAG systems using:
  - 2 retrievers (BM25, E5-Mistral)
  - 4 generators (GPT-4, Llama3, Mixtral, etc.)
- Used a large benchmark dataset across 10 domains
- Designed:
  - Overall metrics (precision, recall, F1)
  - Retriever metrics (claim recall, context precision)
  - Generator metrics (faithfulness, hallucination, etc.)
- Performed meta-evaluation showing strong correlation with human judgment

**What we did in our project**

In our implementation, we adapted the core ideas of the paper with a simpler setup:

- Built 4 RAG systems instead of 8:
  - FAISS + Extractive
  - BM25 + Extractive
  - FAISS + GPT
  - BM25 + GPT
- Used:
  - FAISS as a dense retriever (instead of E5-Mistral)
  - BM25 as a sparse retriever
  - GPT-based and extractive generators
- Evaluated on:
  - Subset of datasets (e.g., NovelQA and CLAPNQ)
  - Smaller sample size (due to computational limits)
- Applied RAGChecker directly as a module:
  - Used its built-in evaluation instead of re-implementing claim extraction and entailment

### **Key Differences**

<br>

| **Aspect**            | **Paper (RAGChecker)**              | **Our Project**                     |
|----------------------|------------------------------------|------------------------------------|
| Number of Systems    | 8                                  | 4                                  |
| Retriever            | BM25 + E5-Mistral (dense)          | BM25 + FAISS (dense)               |
| Generator            | Multiple large LLMs                | GPT-based + Extractive             |
| Dataset Size         | 4000+ queries (10 domains)         | Smaller subset (NovelQA, CLAPNQ)     |
| Evaluation           | RAGChecker Evaluation + Meta-evaluation    | Direct use of RAGChecker module for Evaluation  |

**Table:** Comparison between the original RAGChecker paper and our implementation.